# Mixed Ground and Space Imaging

This minimal example uses three spacecraft: a ground imager, a space-to-space imager, and one passive RSO target. Each observer has its own scenario and reward channel. `MixedReward` keeps the different data types separate while `FreeCommunication` shares knowledge between spacecraft.

BSK-RL and its dependencies must be installed. The fixed geometry and permissive imaging tolerances keep this a short configuration demonstration; they are not mission settings.

In [ ]:
from typing import ClassVar

import numpy as np
from Basilisk.utilities import orbitalMotion

from bsk_rl import ConstellationTasking, act, data, obs, sats, scene
from bsk_rl.comm import FreeCommunication
from bsk_rl.scene.targets import Target
from bsk_rl.sim import dyn, fsw

EPOCH = "2026 OCT 01 00:00:00.000 (UTC)"

## Define the spacecraft

The observers use different dynamics, flight software, observations, and imaging actions. The passive spacecraft is a participant in the RSO scenario but produces no reward data. `FixedGroundTargets` places one ground target below the ground observer after the Earth's orientation is initialized.

In [ ]:
class FixedGroundTargets(scene.UniformTargets):
    """Put a ground target directly below the initial imaging satellite."""

    def regenerate_targets(self):
        self.targets = [
            Target("city", [orbitalMotion.REQ_EARTH * 1e3, 0, 0], priority=3)
        ]

    def reset_post_sim_init(self):
        satellite = self.satellites[0]
        position_P = satellite.dynamics.world.PN @ satellite.dynamics.r_BN_N
        # Access registration retains this array; update it before windows are
        # generated, accounting for the realized Earth orientation at the epoch.
        self.targets[0].r_LP_P[:] = (
            self.radius * position_P / np.linalg.norm(position_P)
        )


class GroundObserver(sats.ImagingSatellite):
    dyn_type = dyn.GroundStationDynModel
    fsw_type = fsw.ImagingFSWModel
    observation_spec: ClassVar[list[obs.Observation]] = [obs.Time()]
    action_spec: ClassVar[list[act.Action]] = [
        act.Image(1, max_duration=30),
        act.Charge(duration=10),
    ]


class RSOObserver(sats.Satellite):
    dyn_type = dyn.SpaceToSpaceImagingDynModel
    fsw_type = fsw.SpaceToSpaceImagingFSWModel
    observation_spec: ClassVar[list[obs.Observation]] = [
        obs.Time(),
        obs.RSOTargetProperties(n_ahead_observe=1),
    ]
    action_spec: ClassVar[list[act.Action]] = [
        act.ImageRSO(
            1,
            max_duration=30,
            min_pointing_hold_s=0,
            require_illumination_during_hold=False,
        ),
        act.Charge(duration=10),
    ]


class PassiveTarget(sats.Satellite):
    dyn_type = dyn.RSOTargetDynModel
    fsw_type = fsw.FSWModel
    observation_spec: ClassVar[list[obs.Observation]] = [obs.Time()]
    action_spec: ClassVar[list[act.Action]] = [act.Drift(duration=1e9)]

## Bind scenarios and reward channels

`MixedScenario.satellite_mapping` lists all participants, including the passive RSO. `MixedReward.satellite_mapping` lists only the satellites producing each reward channel, and `scenario_mapping` binds each channel to its scenario ID. Channel IDs are explicit names, independent of data classes.

The ground target has priority 3 and the RSO has priority 2. The RSO reward callback awards its priority at acquisition for this example; the default RSO reward instead requires complete downlink.

In [ ]:
# Use a nearby RSO and a ground target below the ground observer so that
# this short demonstration can acquire both images.
satellite_args = dict(
    rN=[7e6, 0, 0],
    vN=[0, 7500, 0],
    oe=None,
    sigma_init=[0, 0, 0],
    omega_init=[0, 0, 0],
    wheelSpeeds=[0.0, 0.0, 0.0],
    imageAttErrorRequirement=2.0,  # Permissive pointing tolerance for this demo.
    instrumentBaudRate=1000,
    batteryStorageCapacity=1e9,
    storedCharge_Init=1e9,
    dataStorageCapacity=1e6,
    storageInit=0,
)
satellites = [
    GroundObserver("ground", sat_args=satellite_args),
    RSOObserver("observer", sat_args=satellite_args),
    PassiveTarget("debris"),
]
catalog = scene.RSOTargetCatalog(
    (
        scene.RSOTarget(
            "object", "debris", (7e6 + 1000, 0, 0), (0, 7500, 0), priority=2
        ),
    ),
    EPOCH,
)

mixed_scene = scene.MixedScenario(
    scenarios={
        "ground_scene": FixedGroundTargets(1),
        "rso_scene": scene.RSOTargets(catalog, ["observer"]),
    },
    satellite_mapping={
        "ground_scene": ["ground"],
        "rso_scene": ["observer", "debris"],
    },
)
mixed_reward = data.MixedReward(
    rewarders={
        "ground_images": data.UniqueImageReward(),
        "rso_images": data.RSOImageReward(
            quality_threshold=0,
            acquisition_reward_fn=lambda record, target: target.priority,
        ),
    },
    satellite_mapping={
        "ground_images": ["ground"],
        "rso_images": ["observer"],
    },
    scenario_mapping={
        "ground_images": "ground_scene",
        "rso_images": "rso_scene",
    },
)
env = ConstellationTasking(
    satellites=satellites,
    scenario=mixed_scene,
    rewarder=mixed_reward,
    communicator=FreeCommunication(),
    world_args=dict(utc_init=EPOCH),
    time_limit=60,
    max_step_duration=1,
    sim_rate=1,
    log_level="ERROR",
)

## Run the environment

Both observers image their first candidate. This fixed geometry ensures that the RSO candidate slot is populated; general policies must avoid selecting empty RSO slots.

Expected cumulative rewards are `ground: 3`, `observer: 2`, and `debris: 0`. Every spacecraft learns both channels through communication, but each observer generates data only for its assigned channel. Access heterogeneous data explicitly through `data.data[channel_id]`.

Mappings can overlap to assign several reward channels and scenarios to one satellite, provided that satellite implements the corresponding models and actions. Combining ground and RSO imaging on the same physical satellite requires additional model integration.

In [ ]:
try:
    observations, info = env.reset(seed=0)
    # Action 0 images a target on each observer, and drifts on the passive RSO.
    # None continues the current action on subsequent steps.
    for step in range(8):
        actions = {sat.name: 0 if step == 0 else None for sat in env.satellites}
        observations, rewards, terminated, truncated, info = env.step(actions)

    print("Cumulative rewards:", env.rewarder.cum_reward)
    print("Ground images:", len(env.rewarder.data.data["ground_images"].imaged))
    print("RSO captures:", len(env.rewarder.data.data["rso_images"].captures))
    for sat in env.satellites:
        print(
            sat.name,
            "produces",
            list(sat.data_store.data_stores),
            "knows",
            list(sat.data_store.data.data),
        )
finally:
    env.close()